# Smoking is bad

In [ ]:
# Dependencies are managed by the Pixi smoke environment.
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
data_path = Path("data") / "smoking_data.csv"
print("Working directory:", Path.cwd())
print("CSV found:", data_path.exists())

df = pd.read_csv(data_path)
df.head()

In [ ]:
print("Data dimensions:", df.shape)

In [ ]:
display(df.describe())
display(df.describe(include="all"))

The string columns (`outcome`, `smoker`, and `gender`) are categorical variables, so their summaries show the number of non-missing observations, unique categories, the most frequent category, and its frequency, rather than numerical statistics such as the mean and standard deviation.

In [ ]:
# Check missing values in each column
print("Missing values by column:")
print(df.isna().sum())

print("\nTotal missing values:", df.isna().sum().sum())

In [ ]:
df = df.drop(columns=["Unnamed: 0"], errors="ignore").copy()

df["smokes"] = df["smoker"].map({"No": 0, "Yes": 1})
df["alive"] = df["outcome"].map({"Dead": 0, "Alive": 1})

assert df["smokes"].notna().all(), "Unexpected smoking values found"
assert df["alive"].notna().all(), "Unexpected outcome values found"

print("Prepared data shape:", df.shape)
display(df[["smoker", "smokes", "outcome", "alive", "age", "gender", "salary"]].head())

## Smoking analysis
Let's first check what your cousin did.

In [ ]:
# Overall survival rate by smoking status
survival_summary = df.groupby(["smokes"]).agg(
    prob=("alive", np.mean)
)

display(survival_summary)

print("Sample size by smoking status:")
print(df.groupby("smokes").size())

print("\nOverall sample size:", len(df))

You should get the same results.

# Is smoking really good?

The unadjusted recorded survival proportions were **68.6%** for
non-smokers and **76.1%** for smokers. The
[sample distributions](member_a_distributions.ipynb) describe the age,
smoking status, outcome, gender, and observed salary of the 1,314 records.
Here we compare the two smoking groups within age bands and check whether
the results depend on the band boundaries.

## Baseline age-stratified comparison

Compare smokers and non-smokers within ages **18–29, 30–49, 50–64,
and 65+**. Survival proportion is the count recorded alive divided by
the total for that age band and smoking status. The prepared data
include all 1,314 records because age, smoking status, and outcome
are complete.

In [ ]:
eligible = df.dropna(subset=["age", "smokes", "alive"])
age_band = pd.cut(
    eligible["age"],
    bins=[18, 30, 50, 65, float("inf")],
    labels=["18–29", "30–49", "50–64", "65+"],
    right=False,
)
baseline_survival = (
    eligible.assign(
        age_band=age_band,
        smoking_status=eligible["smokes"].map({0: "Non-smoker", 1: "Smoker"}),
    )
    .groupby(["age_band", "smoking_status"], observed=True)["alive"]
    .agg(survivors="sum", total="size", survival_proportion="mean")
)
assert baseline_survival["total"].sum() == len(eligible)
display(baseline_survival.style.format({"survival_proportion": "{:.1%}"}))

![Recorded survival proportions for smokers and non-smokers in four baseline age bands](figures/member_e_age_stratified_survival.png)

The smokers' survival proportion was lower in each baseline age band.
The largest difference was **8.12 percentage points** at ages 50–64.
Smokers were less represented at older ages: 8.6% were at least 65,
compared with 26.4% of non-smokers. Age is therefore a plausible
confounder of the crude comparison. The 65+ group contains only
50 smokers, including 6 recorded alive.

[Open the full baseline analysis](member_e_age_stratified_survival.ipynb).

## Sensitivity to the choice of age bands

![Survival differences across three age grouping schemes and within age bands at 50 and older](figures/member_c_age_sensitivity.png)

With a common age distribution for the two smoking groups, the
smoker-minus-non-smoker difference was **−4.18 percentage points**
for the baseline bands, **+3.55** for the coarse 50+ scheme, and
**−2.17** for the finer scheme, so the direction of the adjusted
association depends on the age boundaries.

[Open the full sensitivity analysis](member_c_age_sensitivity.ipynb).

These comparisons describe recorded outcomes in this sample.
A causal effect of smoking or a difference in life expectancy
cannot be determined from these comparisons.